# MCTS Deep Dive: From Basics to AlphaGo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmcray/Prometheus_v0_PoC/blob/main/notebooks/mcts_deep_dive.ipynb)

**Goal**: Master Monte Carlo Tree Search (MCTS) - the algorithm that powers AlphaGo and Prometheus.

**Time**: ~45 minutes

**What You'll Learn**:
- What is MCTS and why it works
- The PUCT formula (exploration vs exploitation)
- How to build and visualize MCTS trees
- Direct comparison: MCTS vs no MCTS
- Tuning parameters for optimal performance
- When to use MCTS (and when not to)
- Advanced topics: neural networks, AlphaGo

**Prerequisites**: Basic understanding of tree search and probability

## Setup

If running on Google Colab, install Prometheus first:

In [ ]:
# Colab setup
import sys
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('Prometheus_v0_PoC'):
        print("📥 Cloning Prometheus repository...")
        !git clone https://github.com/pmcray/Prometheus_v0_PoC.git
        print("📦 Installing Prometheus package...")
        !cd Prometheus_v0_PoC && pip install -q -r requirements.txt
        !cd Prometheus_v0_PoC && pip install -q -e .
        print("✅ Installation complete!")
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    # Running locally
    if os.path.exists('prometheus'):
        sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import time

# Prometheus imports
from prometheus.models.go_models import PrometheusGoAgent, RandomGoAgent
from prometheus.environments.go import GoEnvironment
from prometheus.evaluation.benchmark import GoEvaluator
from prometheus.configs import ModelBuilder

print("✓ Imports successful")

---

## Part 1: What is MCTS?

### The Problem

In games like Go or Chess, you can't evaluate all possible moves:
- **Go (19×19)**: ~250 legal moves per position
- **Game length**: ~200 moves
- **Total possibilities**: 250^200 ≈ 10^400 positions (more than atoms in universe!)

**Traditional approaches fail**:
- ❌ **Brute force**: Exponentially too many positions
- ❌ **Pure neural network**: May miss tactical sequences
- ❌ **Random search**: Too inefficient

### The Solution: MCTS

**Monte Carlo Tree Search** intelligently explores the most promising moves:

```
Current Position
    ├─ Move A (visited 100 times, win rate 65%) ← Promising!
    ├─ Move B (visited 50 times, win rate 48%)
    ├─ Move C (visited 5 times, win rate 80%)   ← Uncertain!
    └─ Move D (visited 1 time, win rate 0%)     ← Unexplored!
```

**Key Insight**: Balance **exploitation** (try proven moves) with **exploration** (try uncertain moves).

### The Four Steps of MCTS

1. **Selection**: Start at root, pick most promising child (using PUCT formula)
2. **Expansion**: Add new child node(s) to tree
3. **Simulation**: Play out game randomly (or use neural network)
4. **Backpropagation**: Update all nodes on path with result

Repeat these 4 steps hundreds of times, then pick the best move!

---

## Part 2: The PUCT Formula

**PUCT** (Predictor + Upper Confidence Bound for Trees) decides which move to explore:

$$
\text{PUCT}(a) = Q(a) + c_{\text{puct}} \cdot P(a) \cdot \frac{\sqrt{N_{\text{parent}}}}{1 + N(a)}
$$

Where:
- $Q(a)$ = **average value** of move $a$ (exploitation)
- $P(a)$ = **prior probability** from neural network
- $N(a)$ = **visit count** for move $a$
- $N_{\text{parent}}$ = visit count of parent node
- $c_{\text{puct}}$ = **exploration constant** (typically 1.0)

### Breaking It Down

**Exploitation term**: $Q(a)$
- High if move has won in past
- Encourages playing proven moves

**Exploration term**: $c_{\text{puct}} \cdot P(a) \cdot \frac{\sqrt{N_{\text{parent}}}}{1 + N(a)}$
- High if move has low visit count
- High if neural network thinks move is good ($P(a)$)
- Encourages exploring uncertain moves

Let's see it in action:

In [ ]:
def calculate_puct(Q, P, N_parent, N_child, c_puct=1.0):
    """
    Calculate PUCT score for a move.
    
    Args:
        Q: Average value (-1 to 1)
        P: Prior probability (0 to 1)
        N_parent: Parent visit count
        N_child: Child visit count
        c_puct: Exploration constant
    """
    exploitation = Q
    exploration = c_puct * P * np.sqrt(N_parent) / (1 + N_child)
    return exploitation + exploration

# Example scenario
N_parent = 100  # Parent has been visited 100 times

moves = [
    {'name': 'Move A', 'Q': 0.6, 'P': 0.3, 'N': 50},   # Well-explored, good
    {'name': 'Move B', 'Q': 0.4, 'P': 0.2, 'N': 30},   # Moderate
    {'name': 'Move C', 'Q': 0.8, 'P': 0.1, 'N': 5},    # Few visits, very good
    {'name': 'Move D', 'Q': 0.0, 'P': 0.4, 'N': 0},    # Unvisited, NN likes it
]

print("PUCT Scores:\n" + "=" * 60)
print(f"{'Move':<10} {'Q':>6} {'P':>6} {'N':>4} {'Exploi':>8} {'Explor':>8} {'PUCT':>8}")
print("-" * 60)

for move in moves:
    puct = calculate_puct(move['Q'], move['P'], N_parent, move['N'])
    exploi = move['Q']
    explor = 1.0 * move['P'] * np.sqrt(N_parent) / (1 + move['N'])
    
    print(f"{move['name']:<10} {move['Q']:>6.2f} {move['P']:>6.2f} {move['N']:>4} "
          f"{exploi:>8.3f} {explor:>8.3f} {puct:>8.3f}")

best_move = max(moves, key=lambda m: calculate_puct(m['Q'], m['P'], N_parent, m['N']))
print(f"\n→ MCTS will explore: {best_move['name']}")
print(f"  Reason: Highest PUCT score = {calculate_puct(best_move['Q'], best_move['P'], N_parent, best_move['N']):.3f}")

**Observation**: Move D is chosen despite Q=0 because it's **unexplored** and the **neural network likes it** (P=0.4).

This is the magic of MCTS: it balances trying proven moves with exploring promising unknowns!

---

## Part 3: Building an MCTS Tree

Let's visualize how an MCTS tree grows over time:

In [ ]:
class SimpleMCTSNode:
    """Simple MCTS node for visualization."""
    
    def __init__(self, move_name, prior):
        self.move_name = move_name
        self.prior = prior
        self.visits = 0
        self.total_value = 0.0
        self.children = []
    
    def value(self):
        """Average value."""
        return self.total_value / self.visits if self.visits > 0 else 0.0
    
    def puct(self, parent_visits, c_puct=1.0):
        """Calculate PUCT score."""
        Q = self.value()
        U = c_puct * self.prior * np.sqrt(parent_visits) / (1 + self.visits)
        return Q + U

# Simulate MCTS tree growth
root = SimpleMCTSNode("Root", 1.0)

# Add initial children with neural network priors
moves = [("A", 0.4), ("B", 0.3), ("C", 0.2), ("D", 0.1)]
for move_name, prior in moves:
    root.children.append(SimpleMCTSNode(move_name, prior))

# Simulate 100 MCTS iterations
np.random.seed(42)
snapshots = [10, 25, 50, 100]
snapshot_data = []

for iteration in range(1, 101):
    # Selection: pick child with highest PUCT
    root.visits += 1
    best_child = max(root.children, key=lambda c: c.puct(root.visits))
    
    # Simulation: random outcome
    outcome = np.random.uniform(-1, 1)
    
    # Backpropagation
    best_child.visits += 1
    best_child.total_value += outcome
    
    # Save snapshot
    if iteration in snapshots:
        snapshot_data.append({
            'iteration': iteration,
            'visits': [c.visits for c in root.children],
            'values': [c.value() for c in root.children],
            'puct': [c.puct(root.visits) for c in root.children]
        })

# Visualize tree growth
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
move_names = ["A", "B", "C", "D"]

for idx, snapshot in enumerate(snapshot_data):
    ax = axes[idx]
    
    # Bar plot of visit counts
    bars = ax.bar(move_names, snapshot['visits'], color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'])
    ax.set_title(f"After {snapshot['iteration']} simulations")
    ax.set_ylabel('Visit Count')
    ax.set_ylim(0, max(snapshot['visits']) * 1.2)
    
    # Add value labels
    for i, (bar, visits, value) in enumerate(zip(bars, snapshot['visits'], snapshot['values'])):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f"n={visits}\nQ={value:.2f}",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\nFinal tree state after 100 simulations:")
print(f"{'Move':<6} {'Visits':<8} {'Avg Value':<12} {'PUCT':<8}")
print("-" * 40)
for i, child in enumerate(root.children):
    print(f"{child.move_name:<6} {child.visits:<8} {child.value():<12.3f} {child.puct(root.visits):<8.3f}")

best = max(root.children, key=lambda c: c.visits)
print(f"\n→ Final move choice: {best.move_name} (most visited)")

**Key Observations**:
1. **Early**: MCTS explores all moves to reduce uncertainty
2. **Middle**: Focuses on moves with high priors (A, B)
3. **Late**: Converges on best move based on accumulated evidence
4. **Final decision**: Pick most-visited move (not highest value!)

---

## Part 4: MCTS vs No MCTS

Let's directly compare agent performance with and without MCTS:

In [ ]:
print("Creating agents...\n")

# Create base agent
base_agent = (
    ModelBuilder()
    .go(board_size=9)
    .strength('light')  # Small for fast demo
    .prometheus()
    .build()
)

print(f"Base agent: {base_agent.model.count_params():,} parameters")

# Create MCTS-enhanced version
from prometheus.mcts import add_mcts
mcts_agent = add_mcts(base_agent, num_simulations=200, preset='standard')

print(f"MCTS agent: {mcts_agent.model.count_params():,} parameters + 200 simulations")
print("\nNote: Same neural network, MCTS is the only difference!")

In [ ]:
print("Evaluating agents (10 games each vs random)...\n")

evaluator = GoEvaluator(board_size=9)
env = GoEnvironment(board_size=9)
baseline = RandomGoAgent(board_size=9)

# Test base agent (no MCTS)
print("1. Base agent (no MCTS):")
result_base = evaluator.evaluate_matchup(
    base_agent,
    baseline,
    env,
    num_games=10,
    verbose=False
)
print(f"   Win rate: {result_base['agent1_win_rate']:.1%}")
print(f"   ELO: {result_base['agent1_elo']:.0f}")

# Test MCTS agent
print("\n2. MCTS agent (200 sims):")
result_mcts = evaluator.evaluate_matchup(
    mcts_agent,
    baseline,
    env,
    num_games=10,
    verbose=False
)
print(f"   Win rate: {result_mcts['agent1_win_rate']:.1%}")
print(f"   ELO: {result_mcts['agent1_elo']:.0f}")

# Calculate improvement
elo_gain = result_mcts['agent1_elo'] - result_base['agent1_elo']
wr_gain = result_mcts['agent1_win_rate'] - result_base['agent1_win_rate']

print(f"\n" + "=" * 50)
print(f"MCTS Improvement:")
print(f"  ELO: {elo_gain:+.0f}")
print(f"  Win rate: {wr_gain:+.1%}")
print("=" * 50)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Win rates
ax1.bar(['No MCTS', 'MCTS\n(200 sims)'], 
        [result_base['agent1_win_rate'], result_mcts['agent1_win_rate']],
        color=['#e74c3c', '#2ecc71'])
ax1.set_ylabel('Win Rate')
ax1.set_title('Win Rate vs Random')
ax1.set_ylim(0, 1)
for i, v in enumerate([result_base['agent1_win_rate'], result_mcts['agent1_win_rate']]):
    ax1.text(i, v + 0.02, f"{v:.1%}", ha='center', fontweight='bold')

# ELO
ax2.bar(['No MCTS', 'MCTS\n(200 sims)'],
        [result_base['agent1_elo'], result_mcts['agent1_elo']],
        color=['#e74c3c', '#2ecc71'])
ax2.set_ylabel('ELO Rating')
ax2.set_title('ELO Rating')
for i, v in enumerate([result_base['agent1_elo'], result_mcts['agent1_elo']]):
    ax2.text(i, v + 20, f"{v:.0f}", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n💡 MCTS adds ~{elo_gain:.0f} ELO to the same neural network!")

**Typical Results**: MCTS adds **200-400 ELO** with just 200 simulations!

This is why AlphaGo used MCTS - it makes the neural network much stronger.

---

## Part 5: Tuning MCTS Parameters

### Key Parameters

1. **Number of simulations** - More = stronger but slower
2. **c_puct** - Exploration constant (typically 0.5-2.0)
3. **Temperature** - Randomness in move selection

Let's test different simulation counts:

In [ ]:
print("Testing MCTS with different simulation counts...\n")
print("Note: This may take a few minutes\n")

simulation_counts = [50, 100, 200, 400, 800]
results = []

for sims in simulation_counts:
    print(f"Testing {sims} simulations...")
    
    # Create agent with MCTS
    agent = add_mcts(base_agent, num_simulations=sims)
    
    # Quick evaluation (5 games)
    result = evaluator.evaluate_matchup(
        agent,
        baseline,
        env,
        num_games=5,
        verbose=False
    )
    
    results.append({
        'sims': sims,
        'elo': result['agent1_elo'],
        'win_rate': result['agent1_win_rate']
    })
    
    print(f"  ELO: {result['agent1_elo']:.0f}, Win rate: {result['agent1_win_rate']:.1%}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sims = [r['sims'] for r in results]
elos = [r['elo'] for r in results]
win_rates = [r['win_rate'] for r in results]

# ELO vs simulations
ax1.plot(sims, elos, marker='o', linewidth=2, markersize=10, color='#3498db')
ax1.set_xlabel('Number of Simulations')
ax1.set_ylabel('ELO Rating')
ax1.set_title('Strength vs MCTS Simulations')
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log')

# Win rate vs simulations
ax2.plot(sims, [wr*100 for wr in win_rates], marker='o', linewidth=2, markersize=10, color='#2ecc71')
ax2.set_xlabel('Number of Simulations')
ax2.set_ylabel('Win Rate (%)')
ax2.set_title('Win Rate vs MCTS Simulations')
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log')
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.show()

print("\n📊 Observations:")
print("  - Strength increases with simulations (logarithmic)")
print("  - Diminishing returns: 800 sims not 2x better than 400")
print("  - Sweet spot: 200-400 simulations for most cases")

### Tuning Guidelines

| Simulations | Strength | Use Case | Speed |
|-------------|----------|----------|-------|
| **50-100** | +100-200 ELO | Fast play, testing | Very fast |
| **200-400** | +200-400 ELO | Online play, tournaments | Fast |
| **800-1600** | +300-500 ELO | Strong play, analysis | Medium |
| **3200+** | +400-600 ELO | Maximum strength | Slow |

**Rule of thumb**: Doubling simulations adds ~50-100 ELO

---

## Part 6: Practical Usage in Prometheus

### Easy API

Prometheus makes MCTS simple:

In [ ]:
from prometheus.configs import ModelBuilder
from prometheus.mcts import add_mcts

# Method 1: Built-in presets
agent_fast = (
    ModelBuilder()
    .go(9)
    .strength('medium')
    .mcts('fast')  # 100 simulations
    .prometheus()
    .build()
)

agent_standard = (
    ModelBuilder()
    .go(9)
    .strength('medium')
    .mcts('standard')  # 400 simulations
    .prometheus()
    .build()
)

agent_strong = (
    ModelBuilder()
    .go(9)
    .strength('medium')
    .mcts('strong')  # 800 simulations
    .prometheus()
    .build()
)

# Method 2: Custom simulations
agent_custom = (
    ModelBuilder()
    .go(9)
    .strength('medium')
    .prometheus()
    .build()
)
agent_custom = add_mcts(agent_custom, num_simulations=1600, c_puct=1.5)

print("✓ MCTS agents created with different configurations")

### CLI Usage

```bash
# Train with MCTS
prometheus train --game go --board-size 9 --mcts --mcts-sims 400

# Deploy with MCTS
prometheus deploy --platform ogs --model models/go_9x9.h5 --mcts --mcts-sims 800

# Benchmark MCTS
prometheus benchmark --model models/go_9x9.h5 --metric mcts --mcts-sims 100 400 800
```

---

## Part 7: When to Use MCTS (and When Not To)

### ✅ Use MCTS When:

1. **Playing games** - Especially Go, Chess, Shogi
2. **Tactical sequences** - Need to read ahead
3. **You have time** - MCTS needs computation
4. **Discrete actions** - Clear move choices
5. **Perfect information** - Can simulate game

**Examples**: 
- Go tournaments (use 800+ simulations)
- Chess analysis (use 400+ simulations)
- Game AI (use 200+ simulations)

### ❌ Don't Use MCTS When:

1. **Real-time constraints** - Need instant moves (<100ms)
2. **Continuous actions** - Robotics, control
3. **Partial information** - Card games, fog of war
4. **Random outcomes** - Dice games, poker
5. **Training** - Slows down self-play

**Examples**:
- Bullet chess (too fast)
- Poker (hidden information)
- Self-play training (use policy network directly)

### Hybrid Approach

**Best practice**: 
- **Training**: No MCTS (faster self-play)
- **Deployment**: With MCTS (stronger play)

This is exactly what AlphaGo did!

---

## Part 8: Advanced Topics

### MCTS + Neural Networks (AlphaGo)

**Traditional MCTS**: Random simulations (Monte Carlo)

**AlphaGo MCTS**: Neural network guides search

```
Traditional MCTS:
    Prior P(a) = 1/N (uniform)
    Simulation = random play
    → Needs millions of simulations

AlphaGo MCTS:
    Prior P(a) = PolicyNetwork(state, a)
    Simulation = ValueNetwork(state)
    → Needs only hundreds of simulations!
```

**Why it works**:
1. **Policy network** tells MCTS which moves are likely good
2. **Value network** estimates position without full simulation
3. **MCTS** refines these estimates through tree search

**Result**: Best of both worlds!

### Virtual Loss (Parallel MCTS)

To run MCTS on multiple CPU cores:

```python
# Add "virtual loss" to visited nodes
# Prevents multiple threads from exploring same path

def select_with_virtual_loss(node):
    node.visits += VIRTUAL_LOSS  # Temporarily inflate visits
    child = select_best_child(node)
    return child

def backpropagate_with_virtual_loss(node, value):
    node.visits -= VIRTUAL_LOSS  # Remove virtual loss
    node.visits += 1              # Add real visit
    node.total_value += value
```

**Speedup**: Near-linear with CPU cores (4 cores = 3.5x faster)

### Dirichlet Noise (Exploration)

Add noise to root prior probabilities during self-play:

```python
# Encourage exploration at root
alpha = 0.3  # For Go
epsilon = 0.25

noise = np.random.dirichlet([alpha] * num_actions)
priors = (1 - epsilon) * priors + epsilon * noise
```

**Purpose**: Discover new strategies during training

### Temperature Sampling

Control randomness in move selection:

```python
# Temperature τ controls randomness
τ = 1.0  # Default

visit_counts = [child.visits for child in children]
probabilities = softmax(visit_counts, temperature=τ)
move = np.random.choice(children, p=probabilities)
```

- **τ = 0**: Deterministic (pick most visited)
- **τ = 1**: Proportional to visits
- **τ = ∞**: Uniform random

**Usage**: High temperature early game (exploration), low late game (exploitation)

---

## Summary

### What We Learned

1. **MCTS Basics**
   - 4 steps: Selection, Expansion, Simulation, Backpropagation
   - Balances exploitation vs exploration
   - Works by building a search tree iteratively

2. **PUCT Formula**
   - $Q(a)$ = exploitation (proven moves)
   - $U(a)$ = exploration (uncertain moves)
   - $c_{puct}$ controls the balance

3. **Practical Performance**
   - MCTS adds **200-600 ELO** to same network
   - More simulations = stronger (diminishing returns)
   - 200-400 simulations is sweet spot

4. **Integration**
   - Easy to add: `add_mcts(agent, num_simulations=400)`
   - Multiple presets: 'fast', 'standard', 'strong'
   - CLI support: `--mcts --mcts-sims 800`

5. **When to Use**
   - ✅ Games with discrete actions
   - ✅ When you have computation time
   - ✅ Perfect information scenarios
   - ❌ Real-time constraints
   - ❌ Continuous action spaces

6. **Advanced**
   - Neural networks guide search (AlphaGo)
   - Virtual loss enables parallelization
   - Temperature controls exploration
   - Dirichlet noise for self-play

### Key Takeaways

- **MCTS makes weak agents strong**: Same network, 300+ ELO gain
- **Logarithmic scaling**: Doubling sims ≠ double strength
- **Computation vs strength trade-off**: More sims = stronger but slower
- **AlphaGo's secret**: Neural networks + MCTS = superhuman

### Quick Reference

```python
# Add MCTS to any agent
from prometheus.mcts import add_mcts

# Fast (100 sims)
fast_agent = add_mcts(agent, num_simulations=100, preset='fast')

# Standard (400 sims)
standard_agent = add_mcts(agent, num_simulations=400, preset='standard')

# Strong (800 sims)
strong_agent = add_mcts(agent, num_simulations=800, preset='strong')

# Custom
custom_agent = add_mcts(agent, num_simulations=1600, c_puct=1.5)
```

### Next Steps

1. **Experiment**: Try different simulation counts
2. **Deploy**: Use MCTS in your bots (`prometheus deploy --mcts`)
3. **Tune**: Find optimal parameters for your use case
4. **Learn more**: Read AlphaGo paper, Prometheus source code

---

**Congratulations!** You now understand MCTS from basics to AlphaGo! 🎉

**Further Reading**:
- [AlphaGo Paper](https://www.nature.com/articles/nature16961)
- [AlphaZero Paper](https://arxiv.org/abs/1712.01815)
- [MCTS Survey](https://ieeexplore.ieee.org/document/6145622)